# Synthetic 100-feature benchmark: 30-member extension

Runs only the 30-member backward-selected SVM set for the same three seeds and fixed blind set, then saves raw results for merging into the main notebook.

In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Make the notebook work from either the repository root or validation.
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "mistic" / "svmSet.py").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mistic import combined_rank, cvSet, kernelWrapper, paramSet, score_svc, svmSet

sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
N_SAMPLES = 500
N_FEATURES = 100
N_INFORMATIVE = 10
N_REDUNDANT = 10
SIGNAL_FEATURES = set(range(N_INFORMATIVE + N_REDUNDANT))

X_values, y_values = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=2,
    weights=[0.55, 0.45],
    class_sep=1.0,
    flip_y=0.03,
    shuffle=False,
    random_state=2026,
)
feature_names = [
    *(f"informative_{i:02d}" for i in range(N_INFORMATIVE)),
    *(f"redundant_{i:02d}" for i in range(N_REDUNDANT)),
    *(f"noise_{i:02d}" for i in range(N_FEATURES - N_INFORMATIVE - N_REDUNDANT)),
]
X = pd.DataFrame(X_values, columns=feature_names)
y = pd.Series(y_values, name="class")

MODEL_COUNTS = [1, 3, 5, 10, 20]
BLIND_SET_SEED = 42
INNER_SEEDS = list(range(3))
TEST_SIZE = 0.25
INNER_VALIDATION_SIZE = 0.20
RANK_WEIGHT = 0.75
SELECTION_STRATEGIES = ["backward"]

# A compact grid keeps the full selection-by-size experiment tractable.
C_VALUES = [0.25, 1.0, 4.0]
GAMMA_VALUES = [2.0 ** exponent for exponent in (-9, -7, -5)]

print(f"Samples: {len(X)}, features: {X.shape[1]}")
print(y.value_counts().sort_index())

Samples: 500, features: 100
class
0    275
1    225
Name: count, dtype: int64


In [3]:
def metric_row(y_true, predictions, decision_values):
    # Metrics computed only from the untouched blind-set observations.
    return {
        "roc_auc": roc_auc_score(y_true, decision_values),
        "f1": f1_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "accuracy": accuracy_score(y_true, predictions),
    }


def fit_sklearn_pipeline(X_train, y_train, seed):
    pipeline = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(kernel="rbf", class_weight="balanced")),
    ])
    search = GridSearchCV(
        pipeline,
        param_grid={"svc__C": C_VALUES, "svc__gamma": GAMMA_VALUES},
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=seed),
        n_jobs=-1,
        refit=True,
    )
    return search.fit(X_train, y_train)


def fit_svm_set(X_train, y_train, num_models, seed, selection_strategy):
    # Fit the scaler on this outer-training split only.
    scaler = StandardScaler().fit(X_train)
    X_scaled = scaler.transform(X_train)

    splits = cvSet(X_scaled, np.asarray(y_train))
    splits.classification(
        num_sets=num_models,
        validation_size=INNER_VALIDATION_SIZE,
        random_seed=seed,
    )

    ensemble = svmSet(
        SVC(kernel="precomputed", class_weight="balanced"),
        splits,
        score_method=score_svc(weight=1.0).score,  # tune for ROC AUC
        kernel=kernelWrapper(type="rbf"),
        separate_feature_sets=True,
        separate_parameters=True,
    )
    parameter_grid = [
        paramSet(model={"C": cost}, kernel={"gamma": gamma})
        for cost in C_VALUES
        for gamma in GAMMA_VALUES
    ]
    selection_options = dict(
        parameter_grid=parameter_grid,
        feature_ranker=combined_rank(weight=RANK_WEIGHT).compute,
        set_for_rank="sample",
    )
    if selection_strategy != "backward":
        raise ValueError(f"unknown selection strategy: {selection_strategy}")
    ensemble.greedy_backward_selection(
        reduction_factor=0.1,
        tune_models_each_step=False,
        **selection_options,
    )
    return scaler, ensemble

In [4]:
EXTENSION_MODEL_COUNT = 30
extension_rows = []
X_train, X_blind, y_train, y_blind = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=BLIND_SET_SEED,
)

for seed in INNER_SEEDS:
    scaler, ensemble = fit_svm_set(
        X_train, y_train, EXTENSION_MODEL_COUNT, seed, "backward"
    )
    X_blind_scaled = scaler.transform(X_blind)
    decisions = ensemble.decision_function(X_blind_scaled)
    predictions = ensemble.predict(X_blind_scaled)

    member_predictions = np.column_stack([
        ensemble.predict(X_blind_scaled, model_index=index)
        for index in range(EXTENSION_MODEL_COUNT)
    ])
    pairs = [
        np.mean(member_predictions[:, left] != member_predictions[:, right])
        for left in range(EXTENSION_MODEL_COUNT)
        for right in range(left + 1, EXTENSION_MODEL_COUNT)
    ]
    unified_features = np.asarray(ensemble.unified_features, dtype=int)
    unified_feature_set = set(unified_features)
    ranked_unified = np.asarray([
        feature for feature in ensemble.unified_sorted_features
        if feature in unified_feature_set
    ], dtype=int)
    top_unified_features = ranked_unified[:20]
    selected_signal = len(set(top_unified_features).intersection(SIGNAL_FEATURES))

    extension_rows.append({
        "seed": seed,
        "method": "MiSTIC SVM set",
        "selection_strategy": "backward",
        "num_models": EXTENSION_MODEL_COUNT,
        "num_unified_features": len(unified_features),
        "signal_recall": selected_signal / len(SIGNAL_FEATURES),
        "noise_fraction": 1 - selected_signal / len(top_unified_features),
        "member_disagreement": float(np.mean(pairs)),
        **metric_row(y_blind, predictions, decisions),
    })

    unified_reference = fit_sklearn_pipeline(
        X_train.iloc[:, unified_features], y_train, seed
    )
    extension_rows.append({
        "seed": seed,
        "method": "sklearn on unified features",
        "selection_strategy": "backward",
        "num_models": EXTENSION_MODEL_COUNT,
        "num_unified_features": len(unified_features),
        "signal_recall": selected_signal / len(SIGNAL_FEATURES),
        "noise_fraction": 1 - selected_signal / len(top_unified_features),
        "member_disagreement": 0.0,
        **metric_row(
            y_blind,
            unified_reference.predict(X_blind.iloc[:, unified_features]),
            unified_reference.decision_function(X_blind.iloc[:, unified_features]),
        ),
    })
    print(f"completed seed {seed}, 30-member backward selection")

extension_results = pd.DataFrame(extension_rows)
extension_path = repo_root / "validation/Synthetic100_svmSet_30_results.csv"
extension_results.to_csv(extension_path, index=False)
extension_results

Number of Features: 100, Score: 0.858


Number of Features: 90, Score: 0.862


Number of Features: 81, Score: 0.866


Number of Features: 73, Score: 0.873


Number of Features: 66, Score: 0.872


...

Number of Features: 3, Score: 0.682


Number of Features: 2, Score: 0.592


Number of Features: 1, Score: 0.556


completed seed 2, 30-member backward selection


,seed,method,selection_strategy,num_models,num_unified_features,signal_recall,noise_fraction,member_disagreement,roc_auc,f1,balanced_accuracy,accuracy
0,0,MiSTIC SVM set,backward,30,100,0.65,0.35,0.199761,0.891563,0.714286,0.679477,0.648
1,0,sklearn on unified features,backward,30,100,0.65,0.35,0.000000,0.827122,0.736842,0.759058,0.760
2,1,MiSTIC SVM set,backward,30,92,0.60,0.40,0.167559,0.912267,0.837607,0.850543,0.848
3,1,sklearn on unified features,backward,30,92,0.60,0.40,0.000000,0.861542,0.536585,0.667443,0.696
4,2,MiSTIC SVM set,backward,30,99,0.60,0.40,0.176680,0.888975,0.776978,0.771998,0.752
5,2,sklearn on unified features,backward,30,99,0.60,0.40,0.000000,0.822464,0.725664,0.750129,0.752
